# Single Event Multi-Parameter Loss Scan
## Analyzing combined_product_loss sensitivity to all parameters for one event

In [ ]:
import sys
sys.path.append('..')

from tools.geometry import generate_detector
from tools.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim

import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from scipy.interpolate import interp1d

## Setup Detector and Constants

In [ ]:
# Configuration
default_json_filename = '../config/SK_geom_config.json'
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
TEMPERATURE = 0.10  # Fixed temperature
N_SCAN_POINTS = 31  # Number of scan points per parameter
K = 7
Nphot = 2_00_000

# Speed of light in medium
C_MEDIUM = 0.299792/1.33  # speed of light in medium

# Setup detector
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

# Setup data simulator for generating target events (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=20,
                                      is_data=True, is_calibration=False)

# Setup prediction simulator with fixed temperature (is_data=False)
prediction_simulator = setup_event_simulator(default_json_filename, Nphot, TEMPERATURE, max_sensors_per_cell=8, K=K, is_data=False)

print(f"Number of detectors: {NUM_DETECTORS}")
print(f"Speed of light in medium: {C_MEDIUM}")
print(f"Number of scan points per parameter: {N_SCAN_POINTS}")

# Load ROOT file information
with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

## Detector Parameters

In [ ]:
# Detector parameters
detector_params = (
    jnp.array(50.),           # scatter_length
    jnp.array(0.2),          # reflection_rate
    jnp.array(50.),           # absorption_length
    jnp.array(0.001)         # gumbel_softmax_temp
)

from tools.optimization.optimize import load_optimization_config
config = load_optimization_config('../config/single_ring_optimization_config.json')

# Extract basic configuration
default_json_filename = config['basic_config']['default_json_filename']
data_file = config['basic_config']['data_file']
TEMPERATURE = config['basic_config']['temperature']
#N_EVENTS = config['basic_config']['n_events']
K = config['basic_config']['k']
Nphot = config['basic_config']['nphot']
C_MEDIUM = config['basic_config']['c_medium']

# Extract optimization weights
VERTEX_WEIGHT_SCALE = config['optimization_weights']['vertex_weight_scale']
COUNTS_WEIGHT_SCALE = config['optimization_weights']['counts_weight_scale']
ENERGY_WEIGHT_SCALE = config['optimization_weights']['energy_weight_scale']

## Combined Product Loss Function

In [ ]:
from tools.optimization.losses import energy_loss, counts_loss, direction_time_loss

@jit
def origin_time_loss(origin, detector_positions, true_times, true_q, t0, photosensor_radius=0.25, c_medium=(0.299792/1.33), percentile_threshold=85., scale_w=1e8):
    """Vertex time loss component"""
    distances = jnp.linalg.norm(detector_positions - origin[None, :], axis=1)
    expected_times = (distances - photosensor_radius)/ c_medium
    time_residuals = true_times - expected_times - t0
    
    w1 = jnp.where(true_q>0., true_q, 0.)

    neg_time_res = jnp.where(time_residuals < 0, jnp.abs(time_residuals), 0.0)

    ultra_rel_term = jnp.sum(jnp.abs(neg_time_res*w1)) / (jnp.sum(w1) + 1e-8)+0.075

    pos_time_res = jnp.where(time_residuals > 0, jnp.abs(time_residuals), 0.0)

    w2 = jnp.where(true_q>0., true_q, 0.)
    
    proximity_term =  jnp.sum(jnp.abs(time_residuals*w2**2)) / (jnp.sum(w2) + 1e-8)+0.075

    total_loss = (ultra_rel_term*proximity_term)
 
    return total_loss

def spherical_to_cartesian(theta, phi):
    """Convert spherical angles to Cartesian direction vector"""
    sin_theta = jnp.sin(theta)
    cos_theta = jnp.cos(theta)
    sin_phi = jnp.sin(phi)
    cos_phi = jnp.cos(phi)
    
    return jnp.array([sin_theta * cos_phi, sin_theta * sin_phi, cos_theta])

@jit
def combined_product_loss(params, hit_detector_positions, observed_times, observed_counts, 
                                    true_data, detector_params, key, 
                                    vertex_weight=VERTEX_WEIGHT_SCALE, counts_weight=COUNTS_WEIGHT_SCALE, 
                                    energy_weight=ENERGY_WEIGHT_SCALE):
    """
    Combined loss function: product of vertex loss, counts loss, and energy loss
    
    Args:
        params: [x, y, z, t0, theta, phi, energy] where theta and phi are spherical direction angles
        vertex_weight: scaling factor for vertex loss contribution
        counts_weight: scaling factor for counts loss contribution  
        energy_weight: scaling factor for energy loss contribution
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]

    track_params = (energy, position, jnp.array([theta, phi]))
    #track_params = (energy, position, jax.lax.stop_gradient(jnp.array([theta, phi])))
    simulated_data = prediction_simulator(track_params, detector_params, key)
    simulated_counts = simulated_data[0]
    simulated_time = simulated_data[1]
    
    # Calculate individual loss components
    vertex_loss_val = origin_time_loss(position, hit_detector_positions, observed_times, observed_counts, t0, c_medium=C_MEDIUM)
    counts_loss_val = counts_loss(observed_counts, simulated_counts)
    energy_loss_val = energy_loss(simulated_counts, observed_counts)

    direction = spherical_to_cartesian(theta, phi)
    direction_loss_val = direction_time_loss(position, direction, hit_detector_positions, observed_times, observed_counts, \
                                             t0, c_medium=C_MEDIUM, angle_scale_deg=2.0, inv_scale_w=0.05)
    
    # # Weighted product loss with small offset to avoid zero
    # A = jnp.sqrt((vertex_weight * vertex_loss_val + 1e-6) * (counts_weight * counts_loss_val + 1e-6))
    # B = jnp.sqrt((vertex_weight * vertex_loss_val + 1e-6) * (counts_weight * counts_loss_val + 1e-6) * (direction_loss_val+1e-3))

    # combined_loss = jnp.where(jnp.isnan(B), A, B)

    combined_loss = jnp.sqrt((vertex_weight * vertex_loss_val + 1e-6) * (counts_weight * counts_loss_val + 1e-6))
    #combined_loss = jnp.sqrt(direction_loss_val)#jnp.sqrt((counts_weight * counts_loss_val + 1e-6))
    
    return combined_loss#, (vertex_loss_val, counts_loss_val, energy_loss_val, direction_loss_val)

print("Combined product loss function defined")

## Generic Parameter Scan Function

In [ ]:
def perform_parameter_scan(true_params, true_data, key, param_name, param_idx, scan_range, 
                          true_param_values, use_relative=True, verbose=False):
    """
    Perform 1D scan over a single parameter using combined product loss function

    Args:
        true_params: (energy, position, direction) tuple
        true_data: target data-like event (hit_counts, hit_times)
        key: JAX random key
        param_name: name of parameter being scanned (for display)
        param_idx: index in params array [x, y, z, t0, theta, phi, energy]
        scan_range: range to scan (relative ± or absolute range)
        true_param_values: [x, y, z, t0, theta, phi, energy] array of true values
        use_relative: if True, scan_range is relative to true value; if False, absolute
        verbose: if True, print detailed timing information

    Returns:
        dict with param_values, losses, gradients, and timing info
    """
    t_start_total = time.time()

    # Setup phase
    t_start_setup = time.time()

    # Extract hit times from true_data
    hit_counts, hit_times = true_data

    # Filter for detectors that were hit
    hit_mask = hit_counts > -1
    hit_detector_positions = detector_points[hit_mask]
    observed_times = hit_times[hit_mask]
    observed_charge = hit_counts[hit_mask]

    # Generate scan points
    true_value = true_param_values[param_idx]
    if use_relative:
        param_values = jnp.linspace(true_value - scan_range, true_value + scan_range, N_SCAN_POINTS)
    else:
        param_values = jnp.linspace(scan_range[0], scan_range[1], N_SCAN_POINTS)

    t_setup = time.time() - t_start_setup

    # Define loss and gradient function
    def loss_and_grad_fn(params):
        def loss_fn(p):
            # Need a random key for simulator in combined_product_loss
            loss_key = jax.random.PRNGKey(42)  # Fixed key for consistency
            return combined_product_loss(p, hit_detector_positions, observed_times, observed_charge,
                                        true_data, detector_params, loss_key)
        return value_and_grad(loss_fn)(params)

    # Warmup (JIT compilation) - run once before timing
    t_start_warmup = time.time()
    warmup_params = jnp.array(true_param_values)
    _ = loss_and_grad_fn(warmup_params)
    jax.block_until_ready(_)  # Wait for computation to complete
    t_warmup = time.time() - t_start_warmup

    # Scan loop
    losses = []
    gradients = []
    per_point_times = []

    t_start_scan = time.time()
    for i, param_val in enumerate(param_values):
        t_point_start = time.time()

        # Create parameter vector with modified parameter
        params = jnp.array(true_param_values)
        params = params.at[param_idx].set(param_val)

        # Calculate loss and gradient
        t_loss_grad_start = time.time()
        loss, grad_val = loss_and_grad_fn(params)
        jax.block_until_ready((loss, grad_val))  # Ensure computation completes
        t_loss_grad = time.time() - t_loss_grad_start

        # Extract gradient for this parameter
        param_gradient = grad_val[param_idx]

        losses.append(loss)
        gradients.append(param_gradient)

        t_point = time.time() - t_point_start
        per_point_times.append(t_point)

        if verbose and i == 0:
            print(f"    First point timing: {t_point:.4f}s (loss+grad: {t_loss_grad:.4f}s)")

    t_scan = time.time() - t_start_scan
    t_total = time.time() - t_start_total

    # Timing statistics
    per_point_times = np.array(per_point_times)
    timing_info = {
        'total_time': t_total,
        'setup_time': t_setup,
        'warmup_time': t_warmup,
        'scan_time': t_scan,
        'avg_per_point': np.mean(per_point_times),
    }

    if verbose:
        print(f"\n  {param_name} scan timing:")
        print(f"    Total: {timing_info['total_time']:.4f}s")
        print(f"    Per point (avg): {timing_info['avg_per_point']:.4f}s")

    return {
        'param_name': param_name,
        'param_values': jnp.array(param_values),
        'losses': jnp.array(losses),
        'gradients': jnp.array(gradients),
        'true_value': true_value,
        'n_hit_detectors': jnp.sum(hit_mask),
        'timing': timing_info
    }

print("Generic parameter scan function defined")

## Generate Single Data-Like Event

In [ ]:
# Select entry from ROOT file
entry_idx = 2

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

# Process photon data
photon_origins = photon_data['photon_origins']
photon_directions = photon_data['photon_directions']
photon_times = photon_data['photon_times']
N = len(photon_origins)

# the number 1_000_000 is hard coded also in _simulation_core
padding_size = max(0, 1_000_000-N)

# Pad the origins array (2D array with shape [N,3])
photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                    mode='constant', constant_values=0)

# Pad the directions array with a default unit vector [0,0,1]
default_direction = jnp.array([0.0, 0.0, 1.0])
padding_directions = jnp.tile(default_direction, (padding_size, 1))
if padding_size > 0:
    photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
else:
    photon_data['photon_directions'] = photon_directions

# Pad the times array (1D array with shape [N])
photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                      mode='constant', constant_values=0)

photon_data['N'] = N

# Generate random track parameters
key = jax.random.PRNGKey(44)

# Random position within detector bounds (60% of full volume)
fraction = 0.6
r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
key, _ = jax.random.split(key)
theta_pos = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
z_vert = jax.random.uniform(key, shape=(), minval=-detector.H/2 * fraction, 
                           maxval=detector.H/2 * fraction)
true_position = jnp.array([r_vert * jnp.cos(theta_pos), r_vert * jnp.sin(theta_pos), z_vert])

# Random direction
key, _ = jax.random.split(key)
phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
sin_theta = jnp.sqrt(1 - cos_theta**2)
true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

# Use energy from ROOT file
true_energy = photon_data['energy']  # Fixed energy

# Create particle parameters tuple
true_params = (true_energy, true_position, true_direction)

# Generate data-like event
key, _ = jax.random.split(key)
true_data = jax.lax.stop_gradient(data_simulator(true_params, detector_params, key, photon_data))

# Convert direction to spherical coordinates
from tools.optimization.utils.functions import cartesian_to_spherical
true_theta, true_phi = cartesian_to_spherical(true_direction)

# Use t0=0 as default
true_t0 = 0.0

print(f"\nGenerated single event:")
print(f"  Energy: {true_energy:.2f} MeV")
print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"  Direction (Cartesian): [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
print(f"  Direction (Spherical): theta={true_theta:.3f} rad, phi={true_phi:.3f} rad")
print(f"  t0: {true_t0:.2f} ns")

# Count hit detectors
hit_counts, hit_times = true_data
n_hit = jnp.sum(hit_counts > -1)
print(f"  Hit detectors: {n_hit}")

## Perform Parameter Scans

In [ ]:
# Define true parameter values array: [x, y, z, t0, theta, phi, energy]
true_param_values = [
    float(true_position[0]),  # x
    float(true_position[1]),  # y
    float(true_position[2]),  # z
    float(true_t0),           # t0
    float(true_theta),        # theta
    float(true_phi),          # phi
    float(true_energy)        # energy
]

# Define scan configurations
# Format: (param_name, param_idx, scan_range, use_relative)
scan_configs = [
    ('X', 0, 0.5, True),           # ±0.5 m around true X
    ('Y', 1, 0.5, True),           # ±0.5 m around true Y
    ('Z', 2, 0.5, True),           # ±0.5 m around true Z
    ('t0', 3, 1.0, True),         # ±10 ns around true t0
    ('theta', 4, 0.3, True),       # ±0.3 rad around true theta
    ('phi', 5, 0.3, True),         # ±0.3 rad around true phi
    ('E', 6, 100.0, True),         # ±200 MeV around true energy
]

# Perform all scans
scan_results = []

print(f"\nPerforming {len(scan_configs)} parameter scans...")
print("=" * 60)

for i, (param_name, param_idx, scan_range, use_relative) in enumerate(scan_configs):
    print(f"\nScanning parameter: {param_name} (index {param_idx})")
    
    key, _ = jax.random.split(key)
    result = perform_parameter_scan(
        true_params, 
        true_data, 
        key, 
        param_name, 
        param_idx, 
        scan_range,
        true_param_values,
        use_relative=use_relative,
        verbose=True
    )
    
    scan_results.append(result)

print("\n" + "=" * 60)
print("All parameter scans completed!")

## Visualize All Parameter Scans

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(4, 4, figsize=(10, 8))
axes = axes.flatten()

all_handles = []
all_labels = []

# Plot loss and gradient for each parameter
for i, result in enumerate(scan_results):
    param_name = result['param_name']
    param_values = np.array(result['param_values'])
    losses = np.array(result['losses'])
    gradients = np.array(result['gradients'])
    true_value = result['true_value']
    
    # Plot loss
    ax_loss = axes[2*i]
    line_loss, = ax_loss.plot(param_values, losses, 'b-', linewidth=2, label='Loss')
    true_line_loss = ax_loss.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True Value')
    min_loss_idx = np.argmin(losses)
    min_loss_val = param_values[min_loss_idx]
    min_loss_line = ax_loss.axvline(min_loss_val, color='orange', linestyle=':', alpha=0.7, label='Min Loss')
    ax_loss.plot(min_loss_val, losses[min_loss_idx], 'o', color='orange', markersize=8)
    
    ax_loss.set_title(f'{param_name} - Loss', fontsize=12, fontweight='bold')
    ax_loss.set_xlabel(param_name)
    ax_loss.set_ylabel('Loss')
    ax_loss.grid(True, alpha=0.3)
    
    # Plot gradient
    ax_grad = axes[2*i + 1]
    line_grad, = ax_grad.plot(param_values, gradients, 'g-', linewidth=2, label='Gradient')
    true_line_grad = ax_grad.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True Value')
    zero_grad_line = None
    ax_grad.axhline(0, color='gray', linestyle=':', alpha=0.5)

    if np.min(gradients) <= 0 <= np.max(gradients):
        for j in range(len(gradients) - 1):
            g1, g2 = gradients[j], gradients[j + 1]
            if g1 == 0:
                zero_grad_val = param_values[j]
                break
            elif g1 * g2 < 0:
                x1, x2 = param_values[j], param_values[j + 1]
                zero_grad_val = x1 - g1 * (x2 - x1) / (g2 - g1)
                break
        zero_grad_line = ax_grad.axvline(zero_grad_val, color='purple', linestyle='-.', alpha=0.7, label='Zero Grad')
    
    ax_grad.set_title(f'{param_name} - Gradient', fontsize=12, fontweight='bold')
    ax_grad.set_xlabel(param_name)
    ax_grad.set_ylabel(f'∂Loss/∂{param_name}')
    ax_grad.grid(True, alpha=0.3)

    # Collect handles/labels only once
    if i == 0:
        # Collect all relevant handles and labels
        for item in [line_loss, true_line_loss, min_loss_line, line_grad, true_line_grad]:
            all_handles.append(item)
            all_labels.append(item.get_label())
        if zero_grad_line:
            all_handles.append(zero_grad_line)
            all_labels.append(zero_grad_line.get_label())

# Hide unused subplot
axes[-2].axis('off')
axes[-1].axis('off')

# Create a single, figure-level legend
fig.legend(
    all_handles, all_labels,
    loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.75, 0.2)
)

plt.tight_layout(rect=[0, 0.0, 1, 1])  # Leave space at bottom for legend
plt.show()


## Summary Statistics

In [ ]:
print("=" * 80)
print("SINGLE EVENT MULTI-PARAMETER SCAN SUMMARY")
print("=" * 80)

print(f"\nEvent Details:")
print(f"  Energy: {true_energy:.2f} MeV")
print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"  Direction: theta={true_theta:.3f} rad, phi={true_phi:.3f} rad")
print(f"  Hit detectors: {n_hit}")

print(f"\nParameter Scan Results:")
print("-" * 80)
print(f"{'Parameter':<10} {'True Value':<15} {'Min Loss Value':<15} {'Delta':<12} {'Zero Grad?':<12}")
print("-" * 80)

for result in scan_results:
    param_name = result['param_name']
    param_values = np.array(result['param_values'])
    losses = np.array(result['losses'])
    gradients = np.array(result['gradients'])
    true_value = result['true_value']
    
    # Find minimum loss position
    min_loss_idx = np.argmin(losses)
    min_loss_val = param_values[min_loss_idx]
    delta = min_loss_val - true_value
    
    # Check for zero crossing
    has_zero_crossing = np.min(gradients) <= 0 <= np.max(gradients)
    
    print(f"{param_name:<10} {true_value:<15.4f} {min_loss_val:<15.4f} {delta:<12.4f} {'Yes' if has_zero_crossing else 'No':<12}")

print("-" * 80)

# Timing summary
print(f"\nTiming Summary:")
total_time = sum([result['timing']['total_time'] for result in scan_results])
print(f"  Total scan time: {total_time:.2f}s ({total_time/60:.2f} min)")
for result in scan_results:
    param_name = result['param_name']
    scan_time = result['timing']['total_time']
    avg_per_point = result['timing']['avg_per_point']
    print(f"  {param_name}: {scan_time:.2f}s (avg per point: {avg_per_point:.4f}s)")

print("\n" + "=" * 80)

## Save Results

In [ ]:
# Create output directory if it doesn't exist
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare results dictionary
all_results = {
    'c_medium': C_MEDIUM,
    'temperature': TEMPERATURE,
    'n_scan_points': N_SCAN_POINTS,
    'true_energy': float(true_energy),
    'true_position': [float(x) for x in true_position],
    'true_direction': [float(x) for x in true_direction],
    'true_theta': float(true_theta),
    'true_phi': float(true_phi),
    'true_t0': float(true_t0),
    'n_hit_detectors': int(n_hit),
    'scans': []
}

# Convert scan results to serializable format
for result in scan_results:
    scan_data = {
        'param_name': result['param_name'],
        'param_values': [float(x) for x in result['param_values']],
        'losses': [float(x) for x in result['losses']],
        'gradients': [float(x) for x in result['gradients']],
        'true_value': float(result['true_value']),
        'timing': result['timing']
    }
    all_results['scans'].append(scan_data)

# Save results to pickle file
output_file = output_dir / 'single_event_parameter_scans.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(all_results, f)

print(f"Results saved to {output_file}")